In [1]:
# ============================================
# NOTEBOOK 4: COURSE QUALITY ANALYSIS
# ============================================

# %% [markdown]
# # Course Quality Analysis
# 
# Objectives:
# - Analyze course ratings across categories and levels
# - Examine instructor impact on course quality
# - Identify quality patterns

# %% Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# %% Load Data
df = pd.read_csv('../data/integrated_data.csv')
instructor_metrics = pd.read_csv('../outputs/tables/instructor_metrics.csv')

print(f"✅ Data loaded: {df.shape}")

# %% Course-Level Aggregation
print("\n" + "=" * 60)
print("COURSE-LEVEL METRICS")
print("=" * 60)

course_metrics = df.groupby('CourseID').agg({
    'CourseName': 'first',
    'CourseCategory': 'first',
    'CourseLevel': 'first',
    'CourseRating': 'first',
    'TeacherID': 'first',
    'TeacherName': 'first',
    'TeacherRating': 'first',
    'YearsOfExperience': 'first',
    'Expertise': 'first',
    'TransactionID': 'count'  # Total enrollments
}).reset_index()

course_metrics.columns = [
    'CourseID', 'CourseName', 'CourseCategory', 'CourseLevel',
    'CourseRating', 'TeacherID', 'TeacherName', 'TeacherRating',
    'YearsOfExperience', 'Expertise', 'TotalEnrollments'
]

course_metrics['RatingGap'] = course_metrics['CourseRating'] - course_metrics['TeacherRating']

print(f"\n✅ Course metrics calculated for {len(course_metrics)} courses")
print("\n", course_metrics.head(10))

# Save
course_metrics.to_csv('../outputs/tables/course_metrics.csv', index=False)

# %% Top Rated Courses
print("\n" + "=" * 60)
print("TOP RATED COURSES")
print("=" * 60)

top_courses = course_metrics.nlargest(20, 'CourseRating')[
    ['CourseName', 'CourseCategory', 'CourseLevel', 'CourseRating',
     'TeacherName', 'TeacherRating', 'TotalEnrollments']
]

print("\n🏆 Top 20 Courses:")
print(top_courses.to_string(index=False))

# Visualization
plt.figure(figsize=(14, 10))
plt.barh(range(20), top_courses['CourseRating'], color='gold', edgecolor='black')
plt.yticks(range(20), top_courses['CourseName'], fontsize=9)
plt.xlabel('Course Rating', fontsize=12)
plt.title('Top 20 Highest Rated Courses', fontsize=16, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/top_20_courses.png', dpi=300, bbox_inches='tight')
plt.show()

# %% Most Popular Courses (by enrollment)
print("\n" + "=" * 60)
print("MOST POPULAR COURSES")
print("=" * 60)

popular_courses = course_metrics.nlargest(20, 'TotalEnrollments')[
    ['CourseName', 'CourseCategory', 'CourseLevel', 'CourseRating',
     'TeacherName', 'TotalEnrollments']
]

print("\n📈 Top 20 Most Enrolled Courses:")
print(popular_courses.to_string(index=False))

# %% Course Quality by Category
print("\n" + "=" * 60)
print("COURSE QUALITY BY CATEGORY")
print("=" * 60)

category_analysis = course_metrics.groupby('CourseCategory').agg({
    'CourseRating': ['mean', 'median', 'std', 'min', 'max'],
    'TotalEnrollments': ['sum', 'mean'],
    'CourseID': 'count'
}).round(2)

category_analysis.columns = [
    'AvgRating', 'MedianRating', 'StdRating', 'MinRating', 'MaxRating',
    'TotalEnrollments', 'AvgEnrollments', 'NumCourses'
]

print("\n📊 Quality Metrics by Category:")
print(category_analysis.sort_values('AvgRating', ascending=False))

# Save
category_analysis.to_csv('../outputs/tables/category_analysis.csv')

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Average Rating by Category
category_avg = category_analysis.sort_values('AvgRating', ascending=True)
axes[0, 0].barh(category_avg.index, category_avg['AvgRating'], color='steelblue')
axes[0, 0].set_xlabel('Average Course Rating')
axes[0, 0].set_title('Average Course Rating by Category', fontweight='bold')
axes[0, 0].axvline(course_metrics['CourseRating'].mean(), color='red', 
                    linestyle='--', label='Overall Average')
axes[0, 0].legend()
axes[0, 0].grid(axis='x', alpha=0.3)

# Total Enrollments by Category
category_enroll = category_analysis.sort_values('TotalEnrollments', ascending=True)
axes[0, 1].barh(category_enroll.index, category_enroll['TotalEnrollments'], color='coral')
axes[0, 1].set_xlabel('Total Enrollments')
axes[0, 1].set_title('Total Enrollments by Category', fontweight='bold')
axes[0, 1].grid(axis='x', alpha=0.3)

# Number of Courses by Category
category_count = category_analysis.sort_values('NumCourses', ascending=True)
axes[1, 0].barh(category_count.index, category_count['NumCourses'], color='mediumseagreen')
axes[1, 0].set_xlabel('Number of Courses')
axes[1, 0].set_title('Course Offerings by Category', fontweight='bold')
axes[1, 0].grid(axis='x', alpha=0.3)

# Rating Variability by Category
category_std = category_analysis.sort_values('StdRating', ascending=True)
axes[1, 1].barh(category_std.index, category_std['StdRating'], color='orchid')
axes[1, 1].set_xlabel('Rating Standard Deviation')
axes[1, 1].set_title('Rating Consistency by Category', fontweight='bold')
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/category_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# %% Course Quality by Level
print("\n" + "=" * 60)
print("COURSE QUALITY BY LEVEL")
print("=" * 60)

level_analysis = course_metrics.groupby('CourseLevel').agg({
    'CourseRating': ['mean', 'median', 'std'],
    'TotalEnrollments': ['sum', 'mean'],
    'CourseID': 'count',
    'YearsOfExperience': 'mean'
}).round(2)

level_analysis.columns = [
    'AvgRating', 'MedianRating', 'StdRating',
    'TotalEnrollments', 'AvgEnrollments', 'NumCourses',
    'AvgInstructorExp'
]

print("\n📊 Quality Metrics by Level:")
print(level_analysis)

# Save
level_analysis.to_csv('../outputs/tables/level_analysis.csv')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Rating by Level
level_avg = level_analysis.sort_values('AvgRating')
axes[0].bar(level_avg.index, level_avg['AvgRating'], color=['lightblue', 'skyblue', 'steelblue'])
axes[0].set_ylabel('Average Course Rating')
axes[0].set_title('Average Course Rating by Level', fontweight='bold')
axes[0].axhline(course_metrics['CourseRating'].mean(), color='red', 
                linestyle='--', label='Overall Average')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Enrollments by Level
axes[1].bar(level_avg.index, level_analysis['TotalEnrollments'], 
            color=['lightcoral', 'coral', 'orangered'])
axes[1].set_ylabel('Total Enrollments')
axes[1].set_title('Total Enrollments by Level', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/level_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# %% Heatmap: Category vs Level
print("\n" + "=" * 60)
print("CATEGORY-LEVEL QUALITY HEATMAP")
print("=" * 60)

heatmap_data = course_metrics.pivot_table(
    values='CourseRating',
    index='CourseCategory',
    columns='CourseLevel',
    aggfunc='mean'
)

plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='RdYlGn', 
            center=course_metrics['CourseRating'].mean(),
            linewidths=1, cbar_kws={'label': 'Average Rating'})
plt.title('Course Quality Heatmap: Category vs Level', fontsize=16, fontweight='bold')
plt.xlabel('Course Level', fontsize=12)
plt.ylabel('Course Category', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/figures/category_level_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n📊 Category-Level Quality Matrix:")
print(heatmap_data)

# %% Instructor Impact on Course Quality
print("\n" + "=" * 60)
print("INSTRUCTOR IMPACT ON COURSE QUALITY")
print("=" * 60)
# Create instructor rating tiers
course_metrics['InstructorTier'] = pd.cut(
    course_metrics['TeacherRating'],
    bins=[0, 3.5, 4.0, 4.5, 5.0],
    labels=['Low (≤3.5)', 'Medium (3.5-4.0)', 'High (4.0-4.5)', 'Excellent (>4.5)']
)

tier_analysis = course_metrics.groupby('InstructorTier').agg({
    'CourseRating': ['mean', 'median', 'std'],
    'TotalEnrollments': ['mean', 'sum'],
    'CourseID': 'count'
}).round(2)

tier_analysis.columns = [
    'AvgCourseRating', 'MedianCourseRating', 'StdCourseRating',
    'AvgEnrollments', 'TotalEnrollments', 'NumCourses'
]

print("\n📊 Course Quality by Instructor Rating Tier:")
print(tier_analysis)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Course Rating by Instructor Tier
axes[0].bar(tier_analysis.index, tier_analysis['AvgCourseRating'], 
            color=['#FF6B6B', '#FFA500', '#4ECDC4', '#45B7D1'])
axes[0].set_ylabel('Average Course Rating')
axes[0].set_title('Course Quality by Instructor Rating Tier', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Enrollments by Instructor Tier
axes[1].bar(tier_analysis.index, tier_analysis['AvgEnrollments'], 
            color=['#FF6B6B', '#FFA500', '#4ECDC4', '#45B7D1'])
axes[1].set_ylabel('Average Enrollments per Course')
axes[1].set_title('Enrollment Impact by Instructor Quality', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/figures/instructor_tier_impact.png', dpi=300, bbox_inches='tight')
plt.show()

# %% Correlation: Teacher Rating vs Course Rating
print("\n" + "=" * 60)
print("TEACHER-COURSE RATING CORRELATION")
print("=" * 60)

correlation_coef, p_value = stats.pearsonr(
    course_metrics['TeacherRating'],
    course_metrics['CourseRating']
)

print(f"\n📊 Pearson Correlation:")
print(f"Correlation Coefficient: {correlation_coef:.4f}")
print(f"P-Value: {p_value:.4e}")

if p_value < 0.001:
    print("✅ Highly significant correlation (p < 0.001)")
elif p_value < 0.05:
    print("✅ Significant correlation (p < 0.05)")
else:
    print("❌ No significant correlation")

# Scatter plot
plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    course_metrics['TeacherRating'],
    course_metrics['CourseRating'],
    c=course_metrics['TotalEnrollments'],
    s=100,
    alpha=0.6,
    cmap='viridis',
    edgecolors='black',
    linewidth=0.5
)
plt.colorbar(scatter, label='Total Enrollments')

# Add regression line
z = np.polyfit(course_metrics['TeacherRating'], course_metrics['CourseRating'], 1)
p = np.poly1d(z)
plt.plot(course_metrics['TeacherRating'], 
         p(course_metrics['TeacherRating']), 
         "r--", linewidth=2, label=f'Trend (r={correlation_coef:.3f})')

plt.xlabel('Teacher Rating', fontsize=12)
plt.ylabel('Course Rating', fontsize=12)
plt.title('Teacher Rating vs Course Rating (Correlation Analysis)', 
          fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/teacher_course_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

# %% ANOVA: Course Rating across Categories
print("\n" + "=" * 60)
print("ANOVA: COURSE RATING ACROSS CATEGORIES")
print("=" * 60)

# Prepare data for ANOVA
categories = course_metrics['CourseCategory'].unique()
category_groups = [
    course_metrics[course_metrics['CourseCategory'] == cat]['CourseRating'].values
    for cat in categories
]

# Perform ANOVA
f_stat, p_value = stats.f_oneway(*category_groups)

print(f"\n📊 One-Way ANOVA Results:")
print(f"F-Statistic: {f_stat:.4f}")
print(f"P-Value: {p_value:.4e}")

if p_value < 0.05:
    print("✅ Significant differences exist between categories")
else:
    print("❌ No significant differences between categories")

# %% Quality-Popularity Matrix
print("\n" + "=" * 60)
print("QUALITY-POPULARITY MATRIX")
print("=" * 60)

# Create quadrants
median_rating = course_metrics['CourseRating'].median()
median_enrollment = course_metrics['TotalEnrollments'].median()
course_metrics['Quadrant'] = 'Unknown'
course_metrics.loc[
    (course_metrics['CourseRating'] >= median_rating) & 
    (course_metrics['TotalEnrollments'] >= median_enrollment), 
    'Quadrant'
] = 'Stars (High Quality, High Popularity)'

course_metrics.loc[
    (course_metrics['CourseRating'] >= median_rating) & 
    (course_metrics['TotalEnrollments'] < median_enrollment), 
    'Quadrant'
] = 'Hidden Gems (High Quality, Low Popularity)'

course_metrics.loc[
    (course_metrics['CourseRating'] < median_rating) & 
    (course_metrics['TotalEnrollments'] >= median_enrollment), 
    'Quadrant'
] = 'Popular but Average'

course_metrics.loc[
    (course_metrics['CourseRating'] < median_rating) & 
    (course_metrics['TotalEnrollments'] < median_enrollment), 
    'Quadrant'
] = 'Needs Improvement'

quadrant_counts = course_metrics['Quadrant'].value_counts()
print("\n📊 Course Distribution by Quadrant:")
print(quadrant_counts)

# Visualization
plt.figure(figsize=(14, 10))
colors = {
    'Stars (High Quality, High Popularity)': 'gold',
    'Hidden Gems (High Quality, Low Popularity)': 'lightblue',
    'Popular but Average': 'orange',
    'Needs Improvement': 'lightcoral'
}

for quadrant, color in colors.items():
    mask = course_metrics['Quadrant'] == quadrant
    plt.scatter(
        course_metrics[mask]['TotalEnrollments'],
        course_metrics[mask]['CourseRating'],
        c=color,
        label=quadrant,
        s=100,
        alpha=0.6,
        edgecolors='black',
        linewidth=0.5
    )

plt.axhline(median_rating, color='red', linestyle='--', alpha=0.5)
plt.axvline(median_enrollment, color='red', linestyle='--', alpha=0.5)
plt.xlabel('Total Enrollments', fontsize=12)
plt.ylabel('Course Rating', fontsize=12)
plt.title('Course Quality-Popularity Matrix', fontsize=16, fontweight='bold')
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/quality_popularity_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Save updated course metrics
course_metrics.to_csv('../outputs/tables/course_metrics_full.csv', index=False)

# %% [markdown]
# ## ✅ Key Findings - Course Quality
# 
# 1. Top Categories: [Identify highest-rated categories]
# 2. Level Impact: [Advanced vs Beginner quality]
# 3. Instructor Influence: Strong correlation (r=X.XX) between teacher and course ratings
# 4. Hidden Gems: X courses with high quality but low enrollment
# 5. Improvement Needed: X courses require quality enhancement

print("\n" + "=" * 60)
print("✅ NOTEBOOK 4 COMPLETED SUCCESSFULLY!")
print("=" * 60)


FileNotFoundError: [Errno 2] No such file or directory: '../data/integrated_data.csv'